# Football Analysis - YOLOv8 Fine-Tuning

Fine-tunes YOLOv8 on the football players detection dataset.

**Before running:**
1. `Runtime -> Change runtime type -> T4 GPU`
2. Upload `football-dataset.zip` to Google Drive inside a folder called `football-analysis`

**Output:** `best.pt` saved to Google Drive

## 1 - Check GPU

In [ ]:
!nvidia-smi
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 2 - Install dependencies

In [ ]:
%pip install -q ultralytics

## 3 - Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

import os

DRIVE_DIR = '/content/drive/MyDrive/football-analysis'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive folder: {DRIVE_DIR}')

## 4 - Unzip dataset

In [ ]:
DATASET_ZIP = f'{DRIVE_DIR}/football-dataset.zip'
DATASET_DIR = '/content/dataset'

if os.path.exists(DATASET_ZIP):
    os.makedirs(DATASET_DIR, exist_ok=True)
    !unzip -q "{DATASET_ZIP}" -d "{DATASET_DIR}"
    subdirs = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(f'{DATASET_DIR}/{d}')]
    DATASET_ROOT = f'{DATASET_DIR}/{subdirs[0]}' if len(subdirs) == 1 else DATASET_DIR
    print(f'Dataset ready: {DATASET_ROOT}')
    print(f'Contents: {os.listdir(DATASET_ROOT)}')
else:
    raise FileNotFoundError(f'ZIP not found: {DATASET_ZIP}')

## 5 - Fix dataset paths

In [ ]:
from pathlib import Path

import yaml

data_yaml_path = Path(DATASET_ROOT) / 'data.yaml'
with open(data_yaml_path) as f:
    data = yaml.safe_load(f)

data['train'] = str(Path(DATASET_ROOT) / 'train' / 'images')
data['val']   = str(Path(DATASET_ROOT) / 'valid' / 'images')
data['test']  = str(Path(DATASET_ROOT) / 'test'  / 'images')

fixed_yaml = '/content/football_data.yaml'
with open(fixed_yaml, 'w') as f:
    yaml.dump(data, f)

print('Classes:', data['names'])
print('Train  :', data['train'])
print('Val    :', data['val'])

## 6 - Train

| Setting | Value | Why |
|---------|-------|-----|
| `yolov8s.pt` | small model | good balance for 612 images |
| `freeze=10` | freeze backbone | preserve pretrained features |
| `epochs=100` | 100 passes | enough for convergence |
| `batch=32` | GPU batch | fits comfortably on T4 16GB |

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data=fixed_yaml,
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    project='/content/runs',
    name='football',
    freeze=10,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
)

## 7 - Evaluate on test set

In [ ]:
best_weights = '/content/runs/football/weights/best.pt'
model_best = YOLO(best_weights)

metrics = model_best.val(data=fixed_yaml, split='test')
print(f'mAP50   : {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')

## 8 - Save best.pt to Google Drive

In [ ]:
import shutil

dest = f'{DRIVE_DIR}/best.pt'
shutil.copy(best_weights, dest)
print(f'Saved to Drive: {dest}')
print()
print('Next steps:')
print('  1. Download best.pt from Drive')
print('  2. Place it at: runs/football/weights/best.pt')
print('  3. Update config.yaml:')
print('       weights: runs/football/weights/best.pt')
print('       ball_class_id: 0')
print('       player_class_ids: [1, 2]')
print('       referee_class_id: 3')

## 9 - Visual check on test images (optional)

In [ ]:
from pathlib import Path

from IPython.display import Image, display

test_images = list((Path(DATASET_ROOT) / 'test' / 'images').glob('*.jpg'))[:3]

for img_path in test_images:
    result = model_best.predict(str(img_path), conf=0.3, verbose=False)[0]
    out_path = f'/content/{img_path.stem}_pred.jpg'
    result.save(out_path)
    display(Image(out_path))